In [29]:
using Pkg                # Load Julia's built-in package manager module (Pkg)

Pkg.activate("..")       # Activate the Project.toml located one directory above
                         # (Allows notebooks/ to use the repository root environment)

using Revise             # Development tool that automatically reloads changes
                         # made to files in src/ without restarting Julia

using Horizons3DViewer2  # Load the custom package Horizons3DViewer2
                         # (This loads src/Horizons3DViewer2.jl and its modules)

using EphemerisSources   # Tools for accessing planetary ephemeris data
using AstroTime, Dates   # Time systems and date handling utilities
using GLMakie            # High-performance interactive visualization backend
using ProgressMeter      # Progress bar utilities for long computations
using LinearAlgebra      # Standard linear algebra operations (norm, dot, etc.)

  Activating project at `~/work/Horizons3DViewer2`


In [251]:
using HTTP, Dates, DataFrames

url = "https://spdf.gsfc.nasa.gov/pub/data/isee/isee3/traj/daily/icetrja.asc"
txt = String(HTTP.get(url).body)

rows = NamedTuple[]
for line in eachsplit(txt, '\n')
    s = strip(line)
    isempty(s) && continue
    startswith(s, "#") && continue  # 念のため

    v = split(s)
    length(v) < 8 && continue

    yy     = parse(Int, v[1])

    # DDDは基本整数。万一 "32.0" みたいな表記でも落ちないようにする
    ddd    = occursin('.', v[2]) ? round(Int, parse(Float64, v[2])) : parse(Int, v[2])

    rad_au = parse(Float64, v[3])
    seclat = parse(Float64, v[4])
    seclon = parse(Float64, v[5])
    hellat = parse(Float64, v[6])
    hellon = parse(Float64, v[7])
    hillon = parse(Float64, v[8])

    date = Date(yy, 1, 1) + Day(ddd - 1)

    push!(rows, (date=date, rad_au=rad_au, seclat=seclat, seclon=seclon,
                 hellat=hellat, hellon=hellon, hillon=hillon))
end

df = DataFrame(rows)
first(df, 5)
println(df[  1,1])
println(df[620,1])

1984-01-01
1985-09-11


In [252]:
using LinearAlgebra

# 1 AU [km]（IAU 2012 定義）
const AU_KM = 149_597_870.7

"""
    sph_ecl_to_xyz(r_au, lat_deg, lon_deg)

黄道球座標 (AU, deg) → 直交座標 (km)

r_au   : 距離 [AU]
lat_deg: 黄緯 [deg]
lon_deg: 黄経 [deg]
"""
function sph_ecl_to_xyz(r_au, lat_deg, lon_deg)

    # deg → rad
    lat = deg2rad(lat_deg)
    lon = deg2rad(lon_deg)

    # AU → km
    r = r_au * AU_KM

    # Cartesian
    x = r * cos(lat) * cos(lon)
    y = r * cos(lat) * sin(lon)
    z = r * sin(lat)

    return x, y, z
end

N = nrow(df)
ssc_states = Matrix{Float64}(undef, 3, N)

@inbounds for i in 1:N
    r   = df.rad_au[i]
    lat = df.seclat[i]
    lon = df.seclon[i]
    x, y, z = sph_ecl_to_xyz(r, lat, lon)
    ssc_states[:, i] .= (x, y, z)
end

println(size(ssc_states))
println(ssc_states[:,1])

(3, 5000)
[-2.671676495591874e7, 1.441507641619072e8, 255875.4590664835]


In [ ]:
#=using SPICE
using Dates, LinearAlgebra, Downloads

# ----------------------------
# 1) 回転して J2000 黄道へ
# ----------------------------
N = size(sc_status, 2)
@assert N == nrow(df)

sc_status_j2000 = similar(sc_status)  # 3×N

for i in 1:N
    # icetrja.asc は日次なので、その日の 00:00:00 UTC を代表時刻として採用
    utc = string(df.date[i], "T00:00:00")
    et  = SPICE.str2et(utc)

    R = SPICE.pxform("ECLIPDATE", "ECLIPJ2000", et)  # 3×3
    sc_status_j2000[:, i] = R * sc_status[:, i]
end

# 使う側では sc_status_j2000 が Ecliptic J2000 の 3×N
size(sc_status_j2000)
=#

SpiceError: The frame ECLIPDATE was not recognized as a known reference frame.

In [117]:
using GLMakie

# sc_status は 3×N
x = ssc_states[1, :]
y = ssc_states[2, :]
z = ssc_states[3, :]

f = Figure(size=(900, 700), fontsize=18)
ax = Axis3(
    f[1, 1],
    aspect = :data,
    viewmode = :fit,
    xlabel = "X [AU]",
    ylabel = "Y [AU]",
    zlabel = "Z [AU]",
)

lines!(ax, x, y, z)          # 軌跡
scatter!(ax, x[1], y[1], z[1], markersize=12)  # 始点
scatter!(ax, x[end], y[end], z[end], markersize=12)  # 終点

f

In [234]:
using HTTP          # HTTPリクエストを送信するためのパッケージ
using JSON3         # JSONデータをパース（読み込み）するためのパッケージ
using Dates         # 日付・時刻を扱うための標準ライブラリ

# https://sscweb.gsfc.nasa.gov/WebServices/REST/?utm_source=chatgpt.com
# https://sscweb.gsfc.nasa.gov/WS/sscr/2/observatories

const SSC_BASE = "https://sscweb.gsfc.nasa.gov/WS/sscr/2"  
# NASA SSCWeb (Satellite Situation Center Web) API のベースURL

function ssc_time(dt::DateTime)
    # DateTimeをSSC API用の文字列形式に変換する関数
    
    s = Dates.format(dt, dateformat"yyyymmddHHMMSS")
    # DateTimeを "yyyymmddHHMMSS" 形式の文字列に変換
    
    return s[1:8] * "T" * s[9:end] * "Z"
    # "YYYYMMDDTHHMMSSZ" 形式（ISO風）に整形して返す
end

function ssc_locations_json(observatory::String, t0::DateTime, t1::DateTime;
                            coord::String="geij2000", resolutionFactor::Int=1)
    # 指定した衛星(observatory)の位置データをSSC APIからJSON形式で取得する関数
    # coord: 座標系（デフォルトはGEI J2000）
    # resolutionFactor: データの時間分解能の係数

    tr = string(ssc_time(t0), ",", ssc_time(t1))
    # 開始時刻と終了時刻を「開始,終了」形式の文字列に変換

    url = "$SSC_BASE/locations/$observatory/$tr/$coord/?resolutionFactor=$resolutionFactor"
    # APIリクエスト用URLを構築

    println("URL = ", url)  
    # ★ デバッグ用：実際にアクセスしているURLを表示

    r = HTTP.get(url, ["Accept"=>"application/json", "User-Agent"=>"Julia SSC client"])
    # HTTP GETリクエストを送信
    # AcceptヘッダでJSON形式を要求
    # User-Agentを明示的に指定

    r.status == 200 || error("HTTP $(r.status): $(String(r.body))")
    # HTTPステータスコードが200（成功）でなければエラーを発生

    return JSON3.read(String(r.body))
    # レスポンスボディを文字列化し、JSONとしてパースして返す
end

t0 = DateTime(1981, 11, 1, 0, 0, 0)
# 取得開始時刻（1982年1月1日 00:00:00）

t1 = DateTime(1984, 1, 31, 0, 0, 0)
# 取得終了時刻（1984年1月1日 00:00:00）

data = ssc_locations_json("isee3", t0, t1; coord="geij2000", resolutionFactor=5)
# ISEE-3衛星の位置データをGEI J2000座標系で取得

println(typeof(data))
# 取得したデータの型を表示


URL = https://sscweb.gsfc.nasa.gov/WS/sscr/2/locations/isee3/19811101T000000Z,19840131T000000Z/geij2000/?resolutionFactor=5
JSON3.Array{Any, Base.CodeUnits{UInt8, String}, Vector{UInt64}}


In [235]:

# ------------------------------------------------------------
# SSCWebのレスポンスは
# ["クラス名", 実体]
# という2要素ベクトルでネストされている。
#
# その「型名ラッパー」を取り除くための関数。
# ------------------------------------------------------------
function unwrap(x)

    # x が AbstractVector かどうか確認
    if x isa AbstractVector

        # 要素数が2であるか確認
        if length(x) == 2

            # 1番目の要素が文字列（＝Javaの型名）か確認
            if x[1] isa AbstractString

                # ["型名", 実体] の形式なので、
                # 実体（2番目）を返す
                return x[2]

            end
        end
    end

    # 上記条件に当てはまらない場合は
    # そのまま x を返す
    return x
end

# unwrap はあなたの定義を使用

resp   = unwrap(data)
result = unwrap(resp["Result"])

data2  = unwrap(result["Data"])
sat    = unwrap(data2[1])

coord  = unwrap(unwrap(sat["Coordinates"])[1])

# 位置
X = unwrap(coord["X"])
Y = unwrap(coord["Y"])
Z = unwrap(coord["Z"])
position = hcat(X, Y, Z)'   # 3×N 行列

# 角度
lat = unwrap(coord["Latitude"])
lon = unwrap(coord["Longitude"])

# 時刻（CoordinateDataではなくSatelliteData側）
t_raw = unwrap(sat["Time"])   # 各要素が ["javax.xml.datatype.XMLGregorianCalendar", "1982-..."] みたいな形
t_str = [ unwrap(x) for x in t_raw ]   # "1982-01-01T00:08:00.000+00:00" など

# 地心距離
r = unwrap(sat["RadialLength"])

println(length(position[1,:]))

for i in 1:3
    println(t_str[i])
end
println(t_str[end])

using LinearAlgebra

ε = deg2rad(23.439291111)

R = [
    1  0           0
    0  cos(ε)  sin(ε)
    0 -sin(ε)  cos(ε)
]

R_ecl = R * position

using AstroTime

# SSCの "…+00:00" を外して、from_utc が素直に読める形にする
# 例: "1982-01-01T00:08:00.000"
t_utc_str = replace.(t_str, r"\+00:00$" => "")

# UTC -> TDB（Epoch）へ（うるう秒も考慮される）
t_tdb_ssc = from_utc.(t_utc_str; scale=TDB)

# 確認
println(t_utc_str[1])
println(t_tdb_ssc[1])
println(t_tdb_ssc[end])

19704
1981-11-01T00:08:00.000+00:00
1981-11-01T01:08:00.000+00:00
1981-11-01T02:08:00.000+00:00
1984-01-30T23:32:00.000+00:00
1981-11-01T00:08:00.000
1981-11-01T00:08:52.182 TDB
1984-01-30T23:32:54.184 TDB


In [225]:
using Dates

# ------------------------------------------------------------
# 入力：
#   t_str    : "1982-01-01T00:08:00.000+00:00" のような文字列配列
#   X, Y, Z  : SSCから取り出した各成分（同じ長さ）
#
# 出力：
#   t_new          : 等間隔に整列した DateTime ベクトル
#   position_new   : 補間後の 3×N 行列（各列が [x;y;z]）
#   gaps           : 飛び区間の情報（何番目→何番目が飛んだか等）
# ------------------------------------------------------------

# ① 文字列 -> DateTime (UTC)
fmt = dateformat"yyyy-mm-ddTHH:MM:SS.sss+00:00"
t = DateTime.(t_str, fmt)

# ② 位置 3×N
position = hcat(X, Y, Z)'  # 3×N

# ③ データ長チェック（念のため）
if length(t) != size(position, 2)
    error("length(t) != number of columns in position")
end

# ④ 時刻差分（ミリ秒）
Δt = diff(t)
Δt_ms = Dates.value.(Δt)  # millisecond の整数

# ⑤ 基準ステップを推定（最頻値：mode 相当を自前で計算）
counts = Dict{Int, Int}()
for v in Δt_ms
    if haskey(counts, v)
        counts[v] += 1
    else
        counts[v] = 1
    end
end

base_ms = first(keys(counts))
best_n  = -1
for (k, n) in counts
    if n > best_n
        best_n = n
        base_ms = k
    end
end

base_step = Millisecond(base_ms)
println("Base step ≈ ", base_step)

# ⑥ 「飛び」を検出（基準ステップの 1.5倍より大きい差分はギャップとみなす）
#    ※ 閾値は必要に応じて変更してください
gap_threshold_ms = round(Int, 1.5 * base_ms)

gaps = Vector{NamedTuple{(:i, :t0, :t1, :Δms, :missing_steps), Tuple{Int, DateTime, DateTime, Int, Int}}}()
for i in 1:length(Δt_ms)
    if Δt_ms[i] > gap_threshold_ms
        missing_steps = Int(round(Δt_ms[i] / base_ms)) - 1
        push!(gaps, (i=i, t0=t[i], t1=t[i+1], Δms=Δt_ms[i], missing_steps=missing_steps))
    end
end

println("Number of gaps: ", length(gaps))
for g in gaps
    println("gap at i=", g.i, " : ", g.t0, " -> ", g.t1,
            "  Δ=", Millisecond(g.Δms), "  missing_steps≈", g.missing_steps)
end

# ⑦ 新しい等間隔時刻グリッドを作成
#    最初〜最後まで base_step で刻む
t_new = collect(t[1]:base_step:t[end])

# ⑧ 線形補間で position を新グリッドへ写す（1次補間）
#    ここでは「各軸ごと」に補間します。
function interp1_linear(t_old::Vector{DateTime}, y_old::Vector{Float64}, t_query::Vector{DateTime})
    # 出力配列
    yq = Vector{Float64}(undef, length(t_query))

    # old のインデックス（単調増加前提）
    j = 1

    for i in 1:length(t_query)
        tq = t_query[i]

        # tq が末尾を超えないように調整（保険）
        if tq <= t_old[1]
            yq[i] = y_old[1]
        elseif tq >= t_old[end]
            yq[i] = y_old[end]
        else
            # tq が [t_old[j], t_old[j+1]] に入るまで j を進める
            while !(t_old[j] <= tq <= t_old[j+1])
                j += 1
            end

            t0 = t_old[j]
            t1 = t_old[j+1]
            y0 = y_old[j]
            y1 = y_old[j+1]

            # 線形補間係数（0〜1）
            α = Dates.value(tq - t0) / Dates.value(t1 - t0)

            yq[i] = (1 - α) * y0 + α * y1
        end
    end

    return yq
end

# ⑨ 各成分を補間
X_old = vec(position[1, :])
Y_old = vec(position[2, :])
Z_old = vec(position[3, :])

X_new = interp1_linear(t, X_old, t_new)
Y_new = interp1_linear(t, Y_old, t_new)
Z_new = interp1_linear(t, Z_old, t_new)

# ⑩ 3×N にまとめる
position_new = hcat(X_new, Y_new, Z_new)'  # 3×N

println("Old N = ", length(t), "  New N = ", length(t_new))

Base step ≈ 3600000 milliseconds
Number of gaps: 0
Old N = 16080  New N = 16080


https://juliaastro.org/EphemerisSources.jl/docs/stable/examples/telescope-study/
https://juliaastro.org/AstroTime/stable/tutorial/#Tutorial
https://ssd.jpl.nasa.gov/horizons/app.html#/

In [253]:
######################################
# Specify the data to be retrieved
######################################

frame = "J2000"   # Input parameter for the NASA Horizons API
wrt   = "sun"     # Input parameter for the NASA Horizons API
units = "KM-S"    # Input parameter for the NASA Horizons API

####################
#spacecraft = "Orion EM-1"
#start = TDBEpoch(2022, 11, 16, 9, 4, 0.0)
#stop  = TDBEpoch(2022, 12, 11, 17, 21, 0.0)
#step  = "20 min"
#step_size_sec = 20*60

####################
spacecraft = "ICE"
start = TDBEpoch(1981, 11, 1, 0, 8, 52.182)
stop  = TDBEpoch(1984, 1, 30, 23,32, 54.184)
step  = "1 hour"
step_size_sec = 1*60*60

####################
spacecraft = "ICE"
start = TDBEpoch(1984, 1, 1, 0, 0, 0.0)
stop  = TDBEpoch(1986, 8, 31, 0, 0, 0.0)
step  = "24 hour"
step_size_sec = 24*60*60


####################
#spacecraft = "BepiColombo"                   # Input parameter for the NASA Horizons API
#start = TDBEpoch(2018, 10, 20, 2, 13, 29.0)  # Input parameter for the NASA Horizons API
#stop  = TDBEpoch(2026, 11, 20, 0, 0, 0.0)    # Input parameter for the NASA Horizons API
#step  = "12 hour"                            # Input parameter for the NASA Horizons API
#step_size_sec = 12*60*60                     # Used in the function `propagate_future_trajectories_at_each_epoch_for_animation`

####################
#spacecraft = "BepiColombo"
#start = TDBEpoch(2026, 11, 20, 0, 0, 0.0)
#stop  = TDBEpoch(2027, 4, 7, 0, 1, 9.0)
#step  = "5 min"
#step_size_sec = 5*60

####################
#spacecraft = "Orion EM-1"
#start = TDBEpoch(2022, 11, 16, 9, 3, 0.0)
#stop  = TDBEpoch(2022, 11, 20, 0, 0, 0.0)
#step  = "10 min"
#step_size_sec = 10*60

####################
#spacecraft = "SLIM"
#start = TDBEpoch(2023, 9, 7, 0, 31, 51.0)
#stop  = TDBEpoch(2023, 12, 26, 0, 31, 50.0)
#step  = "15 min"
#step_size_sec = 15*60

####################
#spacecraft = "JUICE"
#start = TDBEpoch(2023, 4, 14, 12, 43, 27.0)
#stop  = TDBEpoch(2031, 7, 21, 6, 31, 30.0)
#step  = "1 day"
#step_size_sec = 24*60*60

####################
#spacecraft = "CAPSTONE"
#start = TDBEpoch(2026, 1, 25, 3, 54, 1.0)
#stop  = TDBEpoch(2026, 3,  3, 11, 36, 27.0)
#stop  = start + 150days
#step  = "10 min"
#step_size_sec = 10*60

####################
#spacecraft = "Hayabusa 2"
#start = TDBEpoch(2014, 12, 3, 6, 14, 54.0)
#stop  = TDBEpoch(2026, 4, 10, 0, 0, 0.0)
#step  = "24 hour"
#step_size_sec = 24*60*60
#ryugu_horizons = ephemeris("162173", start, stop, step ; wrt = wrt, units = units, frame = frame)
#ryugu_states = horizons_to_matrix(ryugu_horizons)

####################
#spacecraft = "Europa Clipper" #全体
#start = TDBEpoch(2024, 10, 14, 16, 15, 4.0)
#stop  = TDBEpoch(2034, 9, 3, 4, 46, 13.0)
#step  = "6 hour"
#step_size_sec = 6*60*60
#europa_horizons = ephemeris("502", start, stop, step ; wrt = wrt, units = units, frame = frame)
#europa_states = horizons_to_matrix(europa_horizons)

####################
#spacecraft = "Europa Clipper" # 木星に入ってから
#start = TDBEpoch(2030, 4, 1, 0, 0, 0.0)
#start = TDBEpoch(2031, 1, 1, 0, 0, 0.0)
#stop  = TDBEpoch(2034, 9, 3, 4, 46, 13.0)
#stop  = TDBEpoch(2032, 1, 1, 0, 0, 0.0)
#step  = "1 hour"
#step_size_sec = 1*60*60
#europa_horizons = ephemeris("502", start, stop, step ; wrt = wrt, units = units, frame = frame)
#europa_states = horizons_to_matrix(europa_horizons)


86400

In [254]:
############################################################################
# Retrieve ephemeris data and convert it into a convenient internal format
############################################################################

# Retrieve ephemeris data from JPL Horizons
sun_horizons     = ephemeris("sun", start, stop, step; wrt = wrt, units = units, frame = frame)
earth_horizons   = ephemeris("earth", start, stop, step; wrt = wrt, units = units, frame = frame)
mars_horizons    = ephemeris("mars", start, stop, step; wrt = wrt, units = units, frame = frame)
jupiter_horizons = ephemeris("jupiter", start, stop, step; wrt = wrt, units = units, frame = frame)
moon_horizons    = ephemeris("moon", start, stop, step; wrt = wrt, units = units, frame = frame)
venus_horizons   = ephemeris("venus", start, stop, step; wrt = wrt, units = units, frame = frame)
mercury_horizons = ephemeris("mercury", start, stop, step; wrt = wrt, units = units, frame = frame)
halley_horizons    = ephemeris("'DES=1P;CAP'", start, stop, step; wrt = wrt, units = units, frame = frame)
giacobini_horizons = ephemeris("'DES=21P;CAP'", start, stop, step; wrt = wrt, units = units, frame = frame)

semb_L1_horizons   = ephemeris("SEMB-L1", start, stop, step; wrt = wrt, units = units, frame = frame)


#sc1_horizons     = ephemeris(spacecraft, start, stop, step; wrt = wrt, units = units, frame = frame)

# Convert the Horizons outputs into 6×N state matrices
sun_states       = horizons_to_matrix(sun_horizons)
earth_states     = horizons_to_matrix(earth_horizons)
mars_states      = horizons_to_matrix(mars_horizons)
jupiter_states   = horizons_to_matrix(jupiter_horizons)
moon_states      = horizons_to_matrix(moon_horizons)
venus_states     = horizons_to_matrix(venus_horizons)
mercury_states   = horizons_to_matrix(mercury_horizons)
halley_states    = horizons_to_matrix(halley_horizons)
giacobini_states = horizons_to_matrix(giacobini_horizons)
semb_L1_states   = horizons_to_matrix(semb_L1_horizons)
#sc1_states       = horizons_to_matrix(sc1_horizons)

# Convert Horizons time stamps (Julian Date) to TDBEpoch values for animation labels
t_tdb = TDBEpoch.(sun_horizons.t .* days; origin = :julian)

# Diagnostic output (for sanity checks)
println(size(earth_states))
println(earth_states[1:6, 1])
println(t_tdb[1])

(6, 974)
[-2.548769060845068e7, 1.448738999263979e8, 5055.889002367854, -29.83676085899024, -5.268940790485321, 0.0008495345194172721]
1984-01-01T00:00:00.000 TDB


In [238]:
println(length(t_tdb))
println(length(t_tdb_ssc))

println(t_tdb[end])
println(t_tdb_ssc[end])

#println(t_tdb[22102:22104])
#println(t_tdb_ssc[22102:22104])


19704
19704
1984-01-30T23:08:52.182 TDB
1984-01-30T23:32:54.184 TDB


In [255]:
#sc1_states = R_ecl[1:3,:]+earth_states[1:3,:]
sc1_states = ssc_states[1:3, 1:length(earth_states[1,:])]

3×974 Matrix{Float64}:
 -2.67168e7  -2.89302e7  -3.14075e7  …   1.50076e8   1.49509e8   1.48905e8
  1.44151e8   1.42197e8   1.4167e8       3.49237e7   3.72767e7   3.96205e7
  2.55875e5   2.53264e5   2.53264e5     -2.6893e5   -2.6893e5   -2.6893e5

In [256]:
######################################
# Heliocentric specific energy
######################################

#sc1_heliocentric_energy     = calc_heliocentric_specific_energy(sc1_states)
#mercury_heliocentric_energy = calc_heliocentric_specific_energy(mercury_states)
#earth_heliocentric_energy   = calc_heliocentric_specific_energy(earth_states)

######################################
# Coordinate transformations
######################################

# Transform states to the EME2000 frame
# (Note: In practice, EME2000 is equivalent to the J2000 mean equator and equinox frame.
#  In SPICE and most astrodynamics applications, the J2000 frame is treated as identical
#  to EME2000, and any differences are negligible for typical spacecraft trajectory analyses.)
#moon_states_eme = eclJ2000_to_EME2000(moon_states - earth_states)
#sc1_states_eme  = eclJ2000_to_EME2000(sc1_states  - earth_states)

# Transform to the Earth–Moon line-fixed rotating frame (Earth-centered)
moon_em_fixed_e = make_line_fixed_frame(earth_states, moon_states, moon_states[1:3,:])
sc1_em_fixed_e  = make_line_fixed_frame(earth_states, moon_states, sc1_states[1:3,:])
sun_em_fixed_e  = make_line_fixed_frame(earth_states, moon_states, sun_states[1:3,:])

# Transform to the Earth–Moon line-fixed rotating frame (Moon-centered)
sc1_em_fixed_m  = sc1_em_fixed_e - moon_em_fixed_e
sun_em_fixed_m  = sun_em_fixed_e - moon_em_fixed_e

# Transform to the Sun–Earth line-fixed rotating frame
earth_se_fixed_s = make_line_fixed_frame(sun_states, earth_states, earth_states[1:3,:])
moon_se_fixed_e  = make_line_fixed_frame(sun_states, earth_states, moon_states[1:3,:]) - earth_se_fixed_s
sc1_se_fixed_e   = make_line_fixed_frame(sun_states, earth_states, sc1_states[1:3,:])  - earth_se_fixed_s
semb_L1_se_fixed_e   = make_line_fixed_frame(sun_states, earth_states, semb_L1_states[1:3,:])  - earth_se_fixed_s

# Transform to the lunar orbital plane–fixed frame (LOP: Lunar Orbital Plane)
# Here, the LOP is defined from the Moon’s orbital plane at the initial epoch and is kept fixed for all subsequent epochs.
moon_lop_e = rotate_to_plane_fixed_t0(earth_states, moon_states, moon_states[1:3,:])
sc1_lop_e  = rotate_to_plane_fixed_t0(earth_states, moon_states, sc1_states[1:3,:])
sun_lop_e  = rotate_to_plane_fixed_t0(earth_states, moon_states, sun_states[1:3,:])

3×974 Matrix{Float64}:
  2.52464e7   2.78106e7   3.03661e7  …  -1.36952e8  -1.37954e8  -1.38917e8
 -1.44814e8  -1.44334e8  -1.43808e8      6.23551e7   6.00122e7   5.76518e7
  5.43471e6   5.65239e6   5.86831e6     -1.35677e7  -1.36054e7  -1.36391e7

In [257]:
######################################
# Animation settings
######################################

# Whether to render the animation
animation_flag = 1

# Length of the trajectory trail
trail_len_long  = 10000
#trail_len_short = 500    # Mio in Sun-centered coordinates
#trail_len_short = 5000  # Mio in Mercury-centered coordinates

trail_len_short = 600    # ICE in Sun-centered coordinates
#trail_len_short = 4800    # ICE in rotationg frame

# Whether to propagate future trajectories for animation
show_propagated_flag = 0

# Animation frame selection
sampling_interval = 25    # Mio in Sun-centered coordinates
sampling_interval = 4    # Mio in Sun-centered coordinates
#sampling_interval = 50  # Mio in Mercury-centered coordinates
N      = size(earth_states, 2)
frames = 1:sampling_interval:(N-1)

# Frame switching (use finer resolution after this point)
#switch_frame = Int(floor(0.8 * N))   # Example: use finer sampling for the last 20%
#frames1 = 1:sampling_interval1:(switch_frame-1)
#frames2 = switch_frame:sampling_interval2:(N-1)
#frames  = vcat(frames1, frames2)

######################################
# Plotting area settings
######################################

set_theme!(theme_black())
update_theme!(fontsize= 18)

fig1 = Figure(size = (900, 600))
fig2 = Figure(size = (900, 600))
fig3 = Figure(size = (900, 600))
fig4 = Figure(size = (900, 600))
fig5 = Figure(size = (900, 600))
fig6 = Figure(size = (900, 600))
fig7 = Figure(size = (900, 600))
fig8 = Figure(size = (900, 600))
fig9 = Figure(size = (900, 600))

ax_ecliptic_s      = default_axis3(fig1[1, 1]; scale=15e7,  title="")
ax_ecliptic_j      = default_axis3(fig2[1, 1]; scale=3e6,  title="Ecliptic of J2000, Jupiter centered")
ax_ecliptic_e      = default_axis3(fig3[1, 1]; scale=3e4,  title="Ecliptic of J2000, Earth centered")
ax_ecliptic_mer    = default_axis3(fig9[1, 1]; scale=2e4,  azimuth = -1.1pi, elevation = 0.05pi, title="Mercury")

ax_lop_e           = default_axis3(fig4[1, 1]; scale=10e5,  title="")#$(spacecraft)")
ax_eme_e           = default_axis3(fig5[1, 1]; scale=2e5,  title="J2000")

ax_em_fixed_m      = default_axis3(fig6[1, 1]; scale=30e3, azimuth = -1.0pi, elevation = -0pi, title="Earth-Moon line fixed rotational frame, Moon centered")
ax_em_fixed_e      = default_axis3(fig7[1, 1]; scale=2e5,  offset=(2e5, 0.0, 0.0), title="Earth-Moon line fixed rotational frame, Earth centered")
ax_se_fixed_e      = default_axis3(fig8[1, 1]; scale=6e5,  title="")#Sun-Earth line fixed rotational frame, Earth centered")

#ax_energy          = Axis(fig1[1, 4], xticks = (1:1, ["Heliocentric energy"]),title = "Heliocentric energy")

#hidedecorations!(ax_energy)
#limits!(ax_energy, (0,2),(0,earth_heliocentric_energy[1]+1250+100))

######################################
# Common plotting routines for both static images and animation
######################################

sun_sphere     = Sphere(Point3f(0.0, 0.0, 0.0), 696340)
earth_sphere   = Sphere(Point3f(0.0, 0.0, 0.0), 6378)
moon_sphere    = Sphere(Point3f(0.0, 0.0, 0.0), 1737)
jupiter_sphere = Sphere(Point3f(0.0, 0.0, 0.0), 69911)
mercury_sphere = Sphere(Point3f(0.0, 0.0, 0.0), 2440)
europa_sphere  = Sphere(Point3f(0.0, 0.0, 0.0), 1600)

# Sun
scatter!(ax_ecliptic_s, 0.0, 0.0, 0.0, markersize=15, color=:white)
text!(ax_ecliptic_s, 0.0, 0.0, 0.0,text = "Sun",color = :white,fontsize = 18)
text!(ax_se_fixed_e, 0.0, 0.0, 0.0,text = "Earth",color = :cyan,fontsize = 18)
text!(ax_lop_e, 0.0, 0.0, 0.0,text = "Earth",color = :cyan,fontsize = 18)

# Earth
mesh!(ax_ecliptic_e, earth_sphere, color=:cyan, shading=true)
mesh!(ax_eme_e, earth_sphere, color=:cyan)
mesh!(ax_lop_e, earth_sphere, color=:cyan, shading=true)
mesh!(ax_se_fixed_e, earth_sphere, color=:cyan)
mesh!(ax_em_fixed_e, earth_sphere, color=:cyan, shading=true)
# Moon
mesh!(ax_em_fixed_m, moon_sphere, color=:yellow, shading=true)
# Jupiter
mesh!(ax_ecliptic_j, jupiter_sphere, color=:orange)
# Mercury
mesh!(ax_ecliptic_mer, mercury_sphere, color=:white, shading=true)

text!(ax_se_fixed_e, 0.95, 0.15, text = "Sun-Earth line fixed rotating frame", offset=(0, 0), align=(:right, :top), space=:relative,fontsize = 16)
text!(ax_lop_e, 0.95, 0.15, text = "Inertial frame", offset=(0, 0), align=(:right, :top), space=:relative,fontsize = 16)
text!(ax_ecliptic_s, 0.95, 0.15, text = "Inertial frame", offset=(0, 0), align=(:right, :top), space=:relative,fontsize = 16)



######################################
# Render static image or animation
######################################

if animation_flag == 0

    lines!(ax_ecliptic_s, earth_states[1:3, :], linewidth=1, color=:cyan)
    lines!(ax_ecliptic_s, halley_states[1:3, :], linewidth=1, color=:yellow)
    lines!(ax_ecliptic_s, giacobini_states[1:3, :], linewidth=1, color=:orange)
    #lines!(ax_ecliptic_s, semb_L1_states[1:3, :], linewidth=1, color=:lime)
    lines!(ax_ecliptic_s, sc1_states[1:3, :],  linewidth=1, color=:lime)

    #=
    lines!(ax_ecliptic_s, earth_states[1:3, :], linewidth=1, color=:cyan)
    #lines!(ax_ecliptic_s, mars_states[1:3, :], linewidth=1, color=:red)
    #lines!(ax_ecliptic_s, jupiter_states[1:3, :], linewidth=1, color=:orange)
    lines!(ax_ecliptic_s, venus_states[1:3, :], linewidth=1, color=:orange)
    lines!(ax_ecliptic_s, mercury_states[1:3, :], linewidth=1, color=:magenta)


    lines!(ax_ecliptic_mer, sc1_states[1:3, :]-mercury_states[1:3, :],  linewidth=1, color=:lime)

    lines!(ax_ecliptic_e, moon_states[1:3, :]- earth_states[1:3, :], linewidth=1, color=:yellow)
    lines!(ax_ecliptic_e, sc1_states[1:3, :]- earth_states[1:3, :],  linewidth=1, color=:lime)
    
    lines!(ax_ecliptic_j, sc1_states[1:3, :]- jupiter_states[1:3, :],  linewidth=1, color=:lime)
    #lines!(ax_ecliptic_j, europa_states[1:3, :]- jupiter_states[1:3, :],  linewidth=1, color=:white)

    #lines!(ax_eme_e, moon_states_eme[1:3, :], linewidth=1, color=:yellow)
    #lines!(ax_eme_e, sc1_states_eme[1:3, :],  linewidth=1, color=:lime)

    lines!(ax_lop_e, moon_lop_e[1:3, :], linewidth=1, color=:yellow)
    lines!(ax_lop_e, sc1_lop_e[1:3, :],  linewidth=1, color=:lime)
    =#

    lines!(ax_se_fixed_e, semb_L1_se_fixed_e[1:3, :], linewidth=1, color=:orange)
    lines!(ax_se_fixed_e, moon_se_fixed_e[1:3, :], linewidth=1, color=:yellow)
    lines!(ax_se_fixed_e, sc1_se_fixed_e[1:3, :],  linewidth=1, color=:lime)
    
    #=
    lines!(ax_em_fixed_e, moon_em_fixed_e[1:3, :], linewidth=1, color=:yellow)
    lines!(ax_em_fixed_e, sc1_em_fixed_e[1:3, :],  linewidth=1, color=:lime)

    lines!(ax_em_fixed_m, sc1_em_fixed_m[1:3, :],  linewidth=1, color=:lime)
    =#
else
    frame_0   = Observable(0)
    frame_0[] = 1

    if show_propagated_flag == 1
        
        results_dict = propagate_future_trajectories_at_each_epoch_for_animation(
            sc1_states, earth_states, moon_states, mars_states, jupiter_states;
            sampling_interval = sampling_interval,
            step_size_sec     = step_size_sec,
            reltol            = 1e-10,
            abstol            = 1e-10,
            show_progress     = true,
        )
        
        lines!(ax_ecliptic_s, @lift(results_dict[$frame_0].sol[1:3, :]),  linewidth=1, color=:red)
        lines!(ax_lop_e, @lift(results_dict[$frame_0].sol_lop_e[1:3, :]),  linewidth=1, color=:red)
        lines!(ax_em_fixed_e, @lift(results_dict[$frame_0].sol_em_fixed_e[1:3, :]),  linewidth=1, color=:red)
        #lines!(ax_em_fixed_m, @lift(results_dict[$frame_0].sol_em_fixed_m[1:3, :]),  linewidth=1, color=:red)
    end

    text!(ax_ecliptic_s, 0.95, 0.85, text = @lift(Dates.format(t_tdb[$frame_0], "yyyy-mm-dd")), offset=(0, 0), align=(:right, :top), space=:relative,fontsize = 24)
    text!(ax_ecliptic_mer, 0.95, 0.85, text = @lift(Dates.format(t_tdb[$frame_0], "yyyy-mm-dd")), offset=(0, 0), align=(:right, :top), space=:relative)
    text!(ax_em_fixed_m, 0.95, 0.85, text = @lift(Dates.format(t_tdb[$frame_0], "yyyy-mm-dd")* " TDB"), offset=(0, 0), align=(:right, :top), space=:relative)
    text!(ax_lop_e, 0.95, 0.85, text = @lift(Dates.format(t_tdb[$frame_0], "yyyy-mm-dd")), offset=(0, 0), align=(:right, :top), space=:relative,fontsize = 24)
    text!(ax_se_fixed_e, 0.95, 0.85, text = @lift(Dates.format(t_tdb[$frame_0], "yyyy-mm-dd")), offset=(0, 0), align=(:right, :top), space=:relative,fontsize = 24)

    text!(ax_ecliptic_s,@lift(Point3f(earth_states[1, $frame_0],earth_states[2, $frame_0],earth_states[3, $frame_0])),text = "Earth",color = :cyan,fontsize = 18)
    text!(ax_ecliptic_s,@lift(Point3f(halley_states[1, $frame_0],halley_states[2, $frame_0],halley_states[3, $frame_0])),text = "Halley",color = :magenta,fontsize = 18)
    text!(ax_ecliptic_s,@lift(Point3f(giacobini_states[1, $frame_0],giacobini_states[2, $frame_0],giacobini_states[3, $frame_0])),text = "Giacobini-Zinner",color = :orange,fontsize = 18)
    #text!(ax_ecliptic_s,@lift(Point3f(venus_states[1, $frame_0],venus_states[2, $frame_0],venus_states[3, $frame_0])),text = "Venus",color = :orange,fontsize = 18)
    #text!(ax_ecliptic_s,@lift(Point3f(mercury_states[1, $frame_0],mercury_states[2, $frame_0],mercury_states[3, $frame_0])),text = "Mercury",color = :white,fontsize = 18)

    #energy
    #barplot!(ax_energy,1,@lift(sc1_heliocentric_energy[$frame_0] + 1250),color = :lime )
    
    #lines!(ax_energy,[0,2],[mercury_heliocentric_energy[1]+1250,mercury_heliocentric_energy[1]+1250],color = :white)
    #lines!(ax_energy,[0,2],[earth_heliocentric_energy[1]+1250,earth_heliocentric_energy[1]+1250],color = :cyan)
    #text!(ax_energy,0.03, mercury_heliocentric_energy[1] + 1250,text = "Mercury",color = :white,fontsize = 15,align = (:left, :bottom))
    #text!(ax_energy,0.03, earth_heliocentric_energy[1] + 1250,  text = "Earth",  color = :cyan, fontsize = 15,align = (:left, :bottom))
    
    traj_fade!(ax_ecliptic_s, earth_states[1:3, :], frame_0; trail_len=trail_len_short, color=:cyan, markersize = 15)
    traj_fade!(ax_ecliptic_s, halley_states[1:3, :], frame_0; trail_len=trail_len_short, color=:magenta, markersize = 15)
    traj_fade!(ax_ecliptic_s, giacobini_states[1:3, :], frame_0; trail_len=trail_len_short, color=:orange, markersize = 15)
    traj_fade!(ax_ecliptic_s, sc1_states[1:3, :], frame_0; trail_len=trail_len_short, color=:lime)

    #=
    traj_fade!(ax_ecliptic_s, earth_states[1:3, :], frame_0; trail_len=trail_len_short, color=:cyan, markersize = 15)
    traj_fade!(ax_ecliptic_s, venus_states[1:3, :], frame_0; trail_len=trail_len_short, color=:orange, markersize = 15)
    traj_fade!(ax_ecliptic_s, mercury_states[1:3, :], frame_0; trail_len=trail_len_short, color=:white, markersize = 15)
    #traj_fade!(ax_ecliptic_s, mars_states[1:3, :], frame_0; trail_len=trail_len_long, color=:pink)
    #traj_fade!(ax_ecliptic_s, jupiter_states[1:3, :], frame_0; trail_len=trail_len_long, color=:orange)
    traj_fade!(ax_ecliptic_s, sc1_states[1:3, :], frame_0; trail_len=trail_len_short, color=:lime)

    traj_fade!(ax_ecliptic_mer, sc1_states[1:3, :] - mercury_states[1:3, :], frame_0; trail_len=trail_len_short, color=:lime, markersize = 0)

    traj_fade!(ax_ecliptic_e, sc1_states[1:3, :]- earth_states[1:3, :], frame_0; trail_len=trail_len_short, color=:lime)
    traj_fade!(ax_ecliptic_j, sc1_states[1:3, :]- jupiter_states[1:3, :], frame_0; trail_len=trail_len_short, color=:lime)
    #traj_fade!(ax_ecliptic_j, europa_states[1:3, :]- jupiter_states[1:3, :], frame_0; trail_len=trail_len_short, color=:white)
    =#

    traj_fade!(ax_lop_e, moon_lop_e, frame_0; trail_len=1200, color=:yellow)    
    traj_fade!(ax_lop_e, sc1_lop_e, frame_0; trail_len=trail_len_short, color=:lime)

    #traj_fade!(ax_em_fixed_e, sc1_em_fixed_e[1:3, :], frame_0; trail_len=trail_len_short, color=:lime)
    #traj_fade!(ax_em_fixed_e, moon_em_fixed_e[1:3, :], frame_0; trail_len=trail_len_short, color=:yellow)

    #traj_fade!(ax_em_fixed_m, sc1_em_fixed_m[1:3, :], frame_0; trail_len=trail_len_short, color=:lime)

    traj_fade!(ax_se_fixed_e, sc1_se_fixed_e[1:3, :], frame_0; trail_len=trail_len_short, color=:lime)
    traj_fade!(ax_se_fixed_e, moon_se_fixed_e[1:3, :], frame_0; trail_len=1200, color=:yellow)
    #traj_fade!(ax_se_fixed_e, semb_L1_se_fixed_e[1:3, :], frame_0; trail_len=trail_len_short, color=:orange)
    
    p = Progress(length(frames); desc = "Rendering movie")

    # Export animation
    record(fig1, "../output/$(spacecraft)_11.mp4", frames; framerate = 30) do frame
        # framerate = number of frames per second (60 fps is typically the practical upper limit)
        # Update planetary frame index for drawing Earth/Mars trajectories

        frame_0[] = frame
        
        # Update solar illumination for each sphere
        set_sun_light!(ax_em_fixed_m, sun_em_fixed_m[1:3, frame])
        set_sun_light!(ax_em_fixed_e, sun_em_fixed_e[1:3, frame])
        set_sun_light!(ax_lop_e, sun_lop_e[1:3, frame])
        set_sun_light!(ax_ecliptic_mer, sun_states[1:3, frame] - mercury_states[1:3, frame])
        
        next!(p)  # Advance progress bar
    end

end

fig1


Rendering movie 100%|████████████████████████████████████| Time: 0:00:03
